<a href="https://colab.research.google.com/github/BernardoBremer/Transfer_learning_actividad/blob/main/transfer_learning_actividad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Instalación de dependencias

In [1]:
# pillow-heif agrega soporte para HEIC/HEIF (formato nativo de iPhone/iPad)
!pip install -q pillow-heif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 40.2 MB/s eta 0:00:00


## 1. Imports y configuracion

In [2]:
import os
import random
import copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder

from PIL import Image
from tqdm import tqdm

# Registrar soporte HEIC/HEIF (fotos de iPhone)
import pillow_heif
pillow_heif.register_heif_opener()

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Dispositivo: cpu


## 2. Cargar Google Drive

In [14]:
from google.colab import drive
drive.mount('/content/drive')

# Ruta al drive
DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks/Inteligencia Computacional/ICO_Transfer_learning')

print('Carpetas detectadas:', sorted([f.name for f in DATA_DIR.iterdir() if f.is_dir()]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpetas detectadas: ['Alex', 'Bremer', 'Edgar', 'Hugo', 'Otros']


## 3. Conversion automatica de imagenes a JPG

Este script recorre todas las subcarpetas y convierte cualquier imagen al formato `.jpg`.  


In [15]:
OK_EXTS  = {'.jpg', '.jpeg'}   # ya estan bien y no se tocan
IMG_EXTS = {'.png', '.webp', '.bmp', '.tiff', '.tif',
            '.gif', '.heic', '.heif', '.avif', '.jfif'}

def convert_images_to_jpg(root: Path):
    converted, skipped, errors = 0, 0, 0

    all_files = sorted(root.rglob('*'))
    for fpath in tqdm(all_files, desc='Revisando archivos'):
        if not fpath.is_file():
            continue

        ext = fpath.suffix.lower()

        if ext in OK_EXTS:
            skipped += 1
            continue

        # Saltar archivos claramente no-imagen
        if ext not in IMG_EXTS and ext not in {'', '.ds_store'}:
            continue

        new_path = fpath.with_suffix('.jpg')
        try:
            img = Image.open(fpath).convert('RGB')
            img.save(new_path, 'JPEG', quality=95)
            fpath.unlink()  # borrar original
            converted += 1
        except Exception as e:
            print(f'  No se pudo convertir {fpath.name}: {e}')
            errors += 1


    print(f'   Ya eran JPG (sin cambios): {skipped}')
    print(f'   Convertidas a JPG:         {converted}')
    print(f'   Errores:                   {errors}')


convert_images_to_jpg(DATA_DIR)

Revisando archivos: 100%|██████████| 250/250 [00:00<00:00, 1686.26it/s]

   Ya eran JPG (sin cambios): 245
   Convertidas a JPG:         0
   Errores:                   0


## 4. EDA  

In [16]:
eda_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

full_dataset = ImageFolder(root=str(DATA_DIR), transform=eda_transform)

CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f'Clases detectadas ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Total de imágenes: {len(full_dataset)}')

class_counts = defaultdict(int)
for _, label in full_dataset.samples:
    class_counts[CLASS_NAMES[label]] += 1

print('\nImagenes por clase:')
for cls, count in class_counts.items():
    print(f'  {cls}: {count}')

Clases detectadas (5): ['Alex', 'Bremer', 'Edgar', 'Hugo', 'Otros']
Total de imágenes: 245

Imagenes por clase:
  Alex: 34
  Bremer: 31
  Edgar: 30
  Hugo: 30
  Otros: 120
